In [1]:
# conda env 'vis2_clone'
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import seaborn as sns

PROJECT_ROOT = Path().resolve().parent.parent.parent
sys.path.append(str(PROJECT_ROOT))

In [2]:
from src.ExperimentSet import ExperimentSet
from src.Retrieval import Retrieval
from src.utils.util import set_labels

In [3]:
EXPERIMENT_SET = PROJECT_ROOT / 'experiments' / 'CAMP2Ex_RF17' / 'V08'
RETRIEVAL = EXPERIMENT_SET / 'default'
FIGURE_OUT = RETRIEVAL / 'figures'
FIGURE_OUT.mkdir(parents=True, exist_ok=True)
r_default = Retrieval(RETRIEVAL)
r_default.load_retrieval(extinction_th=[0.3, np.inf], tau_th=[0.3, np.inf])
savefigs = True

In [4]:
def segment_cloud(cloud, cvm):
    cloud['volume'] = cloud.extinction > 0
    cloud['volume'].data = cloud['volume'].data.astype(np.int64)
    cloud['cth_cloud'] = cloud.volume & cvm.cth_cloud
    cloud['lcl_cloud'] = cloud.volume & cvm.lcl_cloud
    cloud['spc_cloud'] = cloud.volume & cvm.spc_cloud
    cloud['cth_clear'] = cloud.volume & cvm.cth_clear
    cloud['lcl_clear'] = cloud.volume & cvm.lcl_clear
    cloud['spc_clear'] = cloud.volume & cvm.spc_clear
    cloud['cloud'] = cloud.volume & cvm.cloud
    cloud['clear'] = cloud.volume & cvm.clear
    cloud['intersection'] = cloud.cth_cloud & cloud.lcl_cloud & cloud.spc_cloud & cloud.cth_clear & cloud.lcl_clear & cloud.spc_clear & cloud.cloud & cloud.clear
    cloud['union'] = cloud.cth_cloud | cloud.lcl_cloud | cloud.spc_cloud | cloud.cth_clear | cloud.lcl_clear | cloud.spc_clear | cloud.cloud | cloud.clear
    cloud['tau'] = cloud.extinction.integrate(['z'])
    cloud['cloud_mask'] = (cloud.tau > 0).astype(np.int64)
    cloud['cf'] = cloud.cloud_mask.sum() / (cloud.sizes['x'] * cloud.sizes['y'])
    return cloud

In [5]:
# CVM_file = config['cloud_volume_mask_file']

# cvm = xr.load_dataset(config['cloud_volume_mask_file'])
cvm = r_default.cvm
cvm['lcl_mask'].data = cvm['lcl_mask'].data.astype(np.int64)
cvm['cloud_volume_mask'].data = cvm['cloud_volume_mask'].data.astype(np.int64)
cvm['cth_mask'].data[0,:,:] = cvm['cth_mask'].data[-1,:,:] = 0
cvm['cth_mask'].data[:,0,:] = cvm['cth_mask'].data[:,-1,:] = 0
cvm['lcl_mask'].data[0,:,:] = cvm['lcl_mask'].data[-1,:,:] = 0
cvm['lcl_mask'].data[:,0,:] = cvm['lcl_mask'].data[:,-1,:] = 0
cvm['sc_mask'].data[0,:,:] = cvm['sc_mask'].data[-1,:,:] = 0
cvm['sc_mask'].data[:,0,:] = cvm['sc_mask'].data[:,-1,:] = 0
cvm['cloud_volume_mask'].data[0,:,:] = cvm['cloud_volume_mask'].data[-1,:,:] = 0
cvm['cloud_volume_mask'].data[:,0,:] = cvm['cloud_volume_mask'].data[:,-1,:] = 0

cvm['cth_cloud'] = cvm.cth_mask & np.logical_not(cvm.lcl_mask) & np.logical_not(cvm.sc_mask)
cvm['lcl_cloud'] = np.logical_not(cvm.cth_mask) & cvm.lcl_mask & np.logical_not(cvm.sc_mask)
cvm['spc_cloud'] = np.logical_not(cvm.cth_mask) & np.logical_not(cvm.lcl_mask) & cvm.sc_mask
cvm['cth_clear'] = np.logical_not(cvm.cth_mask) & cvm.lcl_mask & cvm.sc_mask
cvm['lcl_clear'] = cvm.cth_mask & np.logical_not(cvm.lcl_mask) & cvm.sc_mask
cvm['spc_clear'] = cvm.cth_mask & cvm.lcl_mask & np.logical_not(cvm.sc_mask)
cvm['cloud'] = cvm.cth_mask & cvm.lcl_mask & cvm.sc_mask
cvm['clear'] = np.logical_not(cvm.cth_mask) & np.logical_not(cvm.lcl_mask) & np.logical_not(cvm.sc_mask)
cvm['clear'].data = cvm['clear'].data.astype(np.int64)
cvm['intersection'] = cvm.cth_cloud & cvm.lcl_cloud & cvm.spc_cloud & cvm.cth_clear & cvm.lcl_clear & cvm.spc_clear & cvm.cloud & cvm.clear
cvm['union'] = cvm.cth_cloud | cvm.lcl_cloud | cvm.spc_cloud | cvm.cth_clear | cvm.lcl_clear | cvm.spc_clear | cvm.cloud | cvm.clear

cvm['cloud_mask'] = (cvm['cloud_volume_mask'].integrate(['z']) > 0).astype(np.int64)
cvm['cf'] = cvm.cloud_mask.sum() / (cvm.sizes['x'] * cvm.sizes['y'])

In [6]:
titles = [
    'default',
    'LCL + CTH',
    'Unconstrained',
    'Dark Surf Unc'
]

r_files = [
    EXPERIMENT_SET / 'default',
    EXPERIMENT_SET / 'CVM' / 'LCL_CTH',
    EXPERIMENT_SET / 'CVM' / 'unconstrained',
    EXPERIMENT_SET / 'CVM' / 'unconstrained_darksurf'
]

clouds = {}
for title, r_file in zip(titles, r_files):
    r_tmp = Retrieval(r_file)
    r_tmp.load_retrieval(extinction_th=[0.3, np.inf], tau_th=[0.3, np.inf])
    cloud = r_tmp.solvers_retrieval[0.67].medium['cloud']
    cloud = segment_cloud(cloud, cvm)
    clouds[title] = cloud

In [8]:
regions = ['cth_cloud', 'lcl_cloud', 'spc_cloud', 'cth_clear', 'lcl_clear', 'spc_clear', 'cloud', 'clear']

# build rows for dataframe
rows = []
for r in regions:
    row = {'Region': r}
    for t in titles:
        c = clouds[t]
        rvol = float(c[r].integrate(['x','y','z']).data)
        tot_vol = float(c.volume.integrate(['x','y','z']).data)
        row[f'{t}_vol'] = rvol
        row[f'{t}_pct'] = (100.0 * rvol / tot_vol) if tot_vol != 0 else np.nan
    row['CVM_vol'] = float(cvm[r].integrate(['x','y','z']).data)
    rows.append(row)

df = pd.DataFrame(rows).set_index('Region')

# add TOTAL row (volumes only, keep pct as NaN)
total_row = {}
for t in titles:
    total_row[f'{t}_vol'] = float(clouds[t].volume.integrate(['x','y','z']).data)
    total_row[f'{t}_pct'] = np.nan
total_row['CVM_vol'] = float(cvm.union.integrate(['x','y','z']).data)

df.loc['TOTAL'] = pd.Series(total_row)

# show resulting dataframe
df

,default_vol,default_pct,LCL + CTH_vol,LCL + CTH_pct,Unconstrained_vol,Unconstrained_pct,Dark Surf Unc_vol,Dark Surf Unc_pct,CVM_vol
Region,,,,,,,,,
cth_cloud,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,12.209656
lcl_cloud,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,22.298031
spc_cloud,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000
cth_clear,0.000000,0.0,0.000000,0.0,7.955750,38.640955,10.935375,37.937820,193.327750
lcl_clear,0.000000,0.0,0.000000,0.0,1.932219,9.384757,3.588406,12.449167,92.716250
spc_clear,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,11.374000
cloud,13.113375,100.0,13.211688,100.0,10.700938,51.974288,14.300688,49.613013,88.163625
clear,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,13.574687
TOTAL,13.113375,NaN,13.211688,NaN,20.588906,NaN,28.824469,NaN,433.664000


In [7]:
regions = ['cth_cloud', 'lcl_cloud', 'spc_cloud', 'cth_clear', 'lcl_clear', 'spc_clear', 'cloud', 'clear']
print('Cloud Volume By Region (pct of total)')
tmp = '{:<10} |'.format('Region')
for t in titles:
    tmp += ' {:<19} |'.format(t)
tmp += ' {:<20}'.format('CVM Volume')
print(tmp)
print('-'*(len(titles)*21 + 12 + 22))
for r in regions:
    tmp = '{:<10} |'.format(r)
    for t in titles:
        c = clouds[t]
        rvol = c[r].integrate(['x','y','z']).data
        tot_vol = c.volume.integrate(['x','y','z']).data
        tmp += ' {:<8.3f} ({:<7.2f}%) |'.format(rvol, 100*rvol/tot_vol)
    tmp += ' {:<20.3f}'.format(cvm[r].integrate(['x','y','z']).data)
    print(tmp)
print('-'*(len(titles)*21 + 12 + 22))
tmp = '{:<10} |'.format('TOTAL')
for t in titles:
    c = clouds[t]
    rvol = c[r].integrate(['x','y','z']).data
    tot_vol = c.volume.integrate(['x','y','z']).data
    tmp += ' {:<19.3f} |'.format(tot_vol)
tmp += ' {:<20.3f}'.format(cvm.union.integrate(['x','y','z']).data)
print(tmp)

Cloud Volume By Region (pct of total)
Region     | default             | LCL + CTH           | Unconstrained       | Dark Surf Unc       | CVM Volume          
----------------------------------------------------------------------------------------------------------------------
cth_cloud  | 0.000    (0.00   %) | 0.000    (0.00   %) | 0.000    (0.00   %) | 0.000    (0.00   %) | 12.210              
lcl_cloud  | 0.000    (0.00   %) | 0.000    (0.00   %) | 0.000    (0.00   %) | 0.000    (0.00   %) | 22.298              
spc_cloud  | 0.000    (0.00   %) | 0.000    (0.00   %) | 0.000    (0.00   %) | 0.000    (0.00   %) | 0.000               
cth_clear  | 0.000    (0.00   %) | 0.000    (0.00   %) | 7.956    (38.64  %) | 10.935   (37.94  %) | 193.328             
lcl_clear  | 0.000    (0.00   %) | 0.000    (0.00   %) | 1.932    (9.38   %) | 3.588    (12.45  %) | 92.716              
spc_clear  | 0.000    (0.00   %) | 0.000    (0.00   %) | 0.000    (0.00   %) | 0.000    (0.00   %) | 11.374    

### Figure: Default & Unconstrained Transects

In [ ]:
ref = 'default'
comp = 'Unconstrained'
ext_cmap = 'gist_heat_r'
x = 4.4
vmax = 35
ticks = [0.0, 4.4, 8.8, 13.2, 17.6]
plt.style.use('seaborn-v0_8-paper')
f, ax = plt.subplots(1, 3, figsize=(14, 3), sharey=True)
clouds[ref].sel(x=x, method='nearest').extinction.plot(y='z', ax=ax[0], vmin=0, vmax=vmax, cmap=ext_cmap)
clouds[comp].sel(x=x, method='nearest').extinction.plot(y='z', ax=ax[1], vmin=0, vmax=vmax, cmap=ext_cmap)
(clouds[comp].extinction - clouds[ref].extinction).sel(x=x, method='nearest').plot(y='z', ax=ax[2], vmin=-20, vmax=20, cmap='RdBu_r')
cvm.sel(x=x, method='nearest').cloud_volume_mask.plot.contour(y='z', levels=1, ax=ax[0], colors='black', linewidths=0.5)
cvm.sel(x=x, method='nearest').cloud_volume_mask.plot.contour(y='z', levels=1, ax=ax[1], colors='black', linewidths=0.5)
cvm.sel(x=x, method='nearest').cloud_volume_mask.plot.contour(y='z', levels=1, ax=ax[2], colors='black', linewidths=0.5)
ax[0].set_title(ref)
ax[1].set_title(comp)
ax[2].set_title('{} - {}'.format(comp, ref))
print(cvm.sel(x=x, method='nearest').x.data)
set_labels(ax.flatten())
for a in ax.flatten():
    a.set_xticks(ticks)

if(savefigs):
    plt.savefig(
        FIGURE_OUT / 'default_unc_transects.png',
        format='png',
        dpi=300,
        bbox_inches='tight',
        transparent=False
    )


In [ ]:
ref = 'default'
comp = 'Dark Surf Unc'
ext_cmap = 'gist_heat_r'
x = 4.4
vmax = 35
plt.style.use('default')
plt.style.use('seaborn-v0_8-paper')
f, ax = plt.subplots(1, 3, figsize=(14, 3), sharey=True)
clouds[ref].sel(x=x, method='nearest').extinction.plot(y='z', ax=ax[0], vmin=0, vmax=vmax, cmap=ext_cmap)
clouds[comp].sel(x=x, method='nearest').extinction.plot(y='z', ax=ax[1], vmin=0, vmax=vmax, cmap=ext_cmap)
(clouds[comp].extinction - clouds[ref].extinction).sel(x=x, method='nearest').plot(y='z', ax=ax[2], vmin=-20, vmax=20, cmap='RdBu_r')
cvm.sel(x=x, method='nearest').cloud_volume_mask.plot.contour(y='z', levels=1, ax=ax[0], colors='black', linewidths=0.5)
cvm.sel(x=x, method='nearest').cloud_volume_mask.plot.contour(y='z', levels=1, ax=ax[1], colors='black', linewidths=0.5)
cvm.sel(x=x, method='nearest').cloud_volume_mask.plot.contour(y='z', levels=1, ax=ax[2], colors='black', linewidths=0.5)

In [ ]:
f, ax = plt.subplots(1, 3, figsize=(15,4))

merged = np.zeros((clouds['Unconstrained'].cloud_mask.sizes['x'], clouds['Unconstrained'].cloud_mask.sizes['y'], 3))
merged[(clouds['Unconstrained'].cloud_mask.data == 0) & (clouds['Dark Surf Unc'].cloud_mask.data == 0)] = [0, 0, 0] # black
merged[(clouds['Unconstrained'].cloud_mask.data == 0) & (clouds['Dark Surf Unc'].cloud_mask.data == 1)] = [1, 0, 0] # red
merged[(clouds['Unconstrained'].cloud_mask.data == 1) & (clouds['Dark Surf Unc'].cloud_mask.data == 0)] = [0, 0, 1] # blue
merged[(clouds['Unconstrained'].cloud_mask.data == 1) & (clouds['Dark Surf Unc'].cloud_mask.data == 1)] = [1, 1, 1] # white
merged_cloud = clouds['Unconstrained'].copy(deep=True)
merged_cloud['merged_cm'] = (['x', 'y', 'rbg'], merged)

(clouds['Unconstrained'].cloud_mask).plot(y='y', ax=ax[0], cmap='gray')
(clouds['Dark Surf Unc'].cloud_mask).plot(y='y', ax=ax[1], cmap='gray')
merged_cloud.merged_cm.plot.imshow(y='y', ax=ax[2], cmap='gray')
ax[0].set_title('Unc {:.3f}'.format(clouds['Unconstrained'].cf.data))
ax[1].set_title('Dark Unc {:.3f}'.format(clouds['Dark Surf Unc'].cf.data))
ax[2].set_title('merged')

In [ ]:
cvm.cf.data